# DockQ打分
对AF3的预测结果运行DockQ，输出docking_results.csv

AF3预测结构链: A(受体), B(肽链)
参考天然结构(DockQ格式 `MODEL:NATIVE`): `AB:{non_L}L`
- A → {non_L} (受体)
- B → L (肽链)


In [ ]:
import os
import subprocess
from pathlib import Path
import pandas as pd
import json
from multiprocessing import Pool
from Bio.PDB import PDBParser

DOCKQ = '/home/junjiechen/miniforge3/envs/dockq/bin/DockQ'
BASE = '/home/junjiechen/1_work/250401-Dpepalign/Benchmark/alphafold3'
METHODS = [
    'msa_notemplate',
    'msa_template',
    'nomsa_notemplate',
    'nomsa_template'
]

REF_DIR = f'{BASE}/dimer'
OUT_BASE = f'{BASE}/predict/original_result'
DOCKQ_OUT = './DockQ'
os.makedirs(DOCKQ_OUT, exist_ok=True)

parser = PDBParser(QUIET=True)

In [9]:
# 生成DockQ命令列表
def get_native_chains(ref_pdb):
    """读取ref_pdb，返回 (non_L_chain, L_chain)"""
    structure = parser.get_structure('ref', ref_pdb)
    chain_ids = [ch.id for ch in structure[0]]
    non_l = [c for c in chain_ids if c != 'L']
    if not non_l:
        raise ValueError(f"No non-L chain found in {ref_pdb}")
    return non_l[0], 'L'


pdbs = [pdb.split('.')[0] for pdb in sorted(os.listdir(REF_DIR)) if pdb.endswith('.pdb')]
with open('./DockQ/dockq.list', 'w') as f:
    for method in METHODS:
        for pdb in pdbs:
            ref_pdb = f'{REF_DIR}/{pdb}.pdb'
            non_l_chain, l_chain = get_native_chains(ref_pdb)            
            pred_dir = f'{OUT_BASE}/{method}/output/{pdb}'
            # if not os.path.isdir(pred_dir):
            #     print(f"Warning: {pred_dir} does not exist, skipping.")
            #     continue
            for seed in ['42', '43', '44']:
                for id in range(5):
                    pred_pdb = f'{pred_dir}/seed-{seed}_sample-{id}/{pdb}_seed-{seed}_sample-{id}_model.cif'   
                    cmd = f'DockQ {pred_pdb} {ref_pdb} --short --mapping AB:{non_l_chain}L'
                    f.write(cmd + '\n')


In [ ]:
# 解析DockQ输出为DataFrame
def parse_dockq_output(method_name):
    """从DockQ输出文件解析结果"""
    out_dir = f'{DOCKQ_OUT}/{method_name}'
    out_file = f'{out_dir}/dockq_results.out'
    
    if not os.path.exists(out_file):
        return None
    
    rows = []
    with open(out_file, 'r') as f:
        lines = f.readlines()
    
    for i, line in enumerate(lines):
        parts = line.strip().split()
        if len(parts) < 20:
            continue
        try:
            dockq_score = float(parts[1])
            irmsd = float(parts[3])
            lrmsd = float(parts[5])
            fnat = float(parts[7])
            
            # 从path中解析pdb, seed, sample
            model_path = parts[16]
            native_path = parts[20]
            
            # 解析模型信息
            # path: .../seed-42_sample-0/1a0n_seed-42_sample-0_model.cif
            cif_name = model_path.split('/')[-1]  # 1a0n_seed-42_sample-0_model.cif
            native_name = native_path.split('/')[-1].replace('.pdb', '')  # 1a0n
            
            # 从cif名中提取seed
            # 1a0n_seed-42_sample-0_model.cif -> seed=42, sample=0
            parts_name = cif_name.replace('_model.cif', '').split('_')
            pdb = parts_name[0]
            seed = None
            sample = None
            for p in parts_name:
                if p.startswith('seed-'):
                    seed = int(p.replace('seed-', ''))
                elif p.startswith('sample-'):
                    sample = int(p.replace('sample-', ''))
            
            rows.append({
                'native': native_name,
                'pdb': pdb,
                'seed': seed,
                'sample': sample,
                'dockq_score': dockq_score,
                'fnat': fnat,
                'lrmsd': lrmsd,
                'irmsd': irmsd,
            })
        except (ValueError, IndexError):
            continue
    
    return pd.DataFrame(rows) if rows else None

# 解析所有方法的DockQ输出并保存
all_results = {}
for method_name in METHODS.values():
    df = parse_dockq_output(method_name)
    if df is not None:
        df.to_csv(f'{DOCKQ_OUT}/{method_name}/docking_results.csv', index=False)
        print(f'{method_name}: {len(df)} rows, {df["native"].nunique()} complexes')
    else:
        print(f'{method_name}: no results (run DockQ first)')